# Оценка моделей ИИ: оценка кода, человека и модели на основе кода

В этом блокноте мы углубимся в три широко используемых метода оценки эффективности моделей ИИ, таких как Claude v3:

1. Оценка на основе кода
2. Человеческая оценка
3. Оценка на основе модели

Мы проиллюстрируем каждый подход на примерах и рассмотрим их преимущества и ограничения при оценке производительности ИИ.

## Пример оценки на основе кода: анализ настроений

В этом примере мы оценим способность Клода классифицировать настроение рецензий на фильмы как положительное или отрицательное. Мы можем использовать код, чтобы проверить, соответствуют ли выходные данные модели ожидаемому настроению.

In [ ]:
# Import python's built-in regular expression library
import re

# Import boto3 and json
import boto3
import json

# Store the model name and AWS region for later use
MODEL_NAME = "anthropic.claude-3-haiku-20240307-v1:0"
AWS_REGION = "us-west-2"

%store MODEL_NAME
%store AWS_REGION

In [ ]:
# Function to build the input prompt for sentiment analysis
def build_input_prompt(review):
    user_content = f"""Classify the sentiment of the following movie review as either 'positive' or 'negative' provide only one of those two choices:
    <review>{review}</review>"""
    return [{"role": "user", "content": user_content}]

# Define the evaluation data
eval = [
    {
        "review": "This movie was amazing! The acting was superb and the plot kept me engaged from start to finish.",
        "golden_answer": "positive"
    },
    {
        "review": "I was thoroughly disappointed by this film. The pacing was slow and the characters were one-dimensional.",
        "golden_answer": "negative"
    }
]

# Function to get completions from the model
client = boto3.client('bedrock-runtime',region_name=AWS_REGION)

def get_completion(messages):
    body = json.dumps(
        {
            "anthropic_version": '',
            "max_tokens": 2000,
            "messages": messages,
            "temperature": 0.5,
            "top_p": 1,
        }
    )
    response = client.invoke_model(body=body, modelId=MODEL_NAME)
    response_body = json.loads(response.get('body').read())

    return response_body.get('content')[0].get('text')

# Get completions for each input
outputs = [get_completion(build_input_prompt(item["review"])) for item in eval]

# Print the outputs and golden answers
for output, question in zip(outputs, eval):
    print(f"Review: {question['review']}\nGolden Answer: {question['golden_answer']}\nOutput: {output}\n")

# Function to grade the completions
def grade_completion(output, golden_answer):
    return output.lower() == golden_answer.lower()

# Grade the completions and print the accuracy
grades = [grade_completion(output, item["golden_answer"]) for output, item in zip(outputs, eval)]
print(f"Accuracy: {sum(grades) / len(grades) * 100}%")

## Пример человеческой оценки: оценка эссе

Некоторые задачи, такие как оценка эссе, сложно оценить с помощью одного лишь кода. В этом случае мы можем предоставить оценщикам рекомендации по оценке результатов модели.

In [ ]:
# Function to build the input prompt for essay generation
def build_input_prompt(topic):
    user_content = f"""Write a short essay discussing the following topic:
    <topic>{topic}</topic>"""
    return [{"role": "user", "content": user_content}]

# Define the evaluation data
eval = [
    {
        "topic": "The importance of education in personal development and societal progress",
        "golden_answer": "A high-scoring essay should have a clear thesis, well-structured paragraphs, and persuasive examples discussing how education contributes to individual growth and broader societal advancement."
    }
]

# Get completions for each input
outputs = [get_completion(build_input_prompt(item["topic"])) for item in eval]

# Print the outputs and golden answers
for output, item in zip(outputs, eval):
    print(f"Topic: {item['topic']}\n\nGrading Rubric:\n {item['golden_answer']}\n\nModel Output:\n{output}\n")

## Примеры оценки на основе модели

Мы можем использовать Claude для оценки собственных результатов, предоставляя ответ модели и критерии оценки. Это позволяет нам автоматизировать оценку задач, которые обычно требуют человеческого решения.

### Пример 1: суммирование

В этом примере мы будем использовать Claude для оценки качества сгенерированного им резюме. Это может быть полезно, когда вам нужно оценить способность модели кратко и точно извлекать ключевую информацию из более длинного текста. Предоставляя рубрику, в которой излагаются основные моменты, которые следует охватить, мы можем автоматизировать процесс оценки и быстро оценить эффективность модели при выполнении задач обобщения.

In [ ]:
# Function to build the input prompt for summarization
def build_input_prompt(text):
    user_content = f"""Please summarize the main points of the following text:
    <text>{text}</text>"""
    return [{"role": "user", "content": user_content}]

# Function to build the grader prompt for assessing summary quality
def build_grader_prompt(output, rubric):
    user_content = f"""Assess the quality of the following summary based on this rubric:
    <rubric>{rubric}</rubric>
    <summary>{output}</summary>
    Provide a score from 1-5, where 1 is poor and 5 is excellent."""
    return [{"role": "user", "content": user_content}]

# Define the evaluation data
eval = [
    {
        "text": "The Magna Carta, signed in 1215, was a pivotal document in English history. It limited the powers of the monarchy and established the principle that everyone, including the king, was subject to the law. This laid the foundation for constitutional governance and the rule of law in England and influenced legal systems worldwide.",
        "golden_answer": "A high-quality summary should concisely capture the key points: 1) The Magna Carta's significance in English history, 2) Its role in limiting monarchical power, 3) Establishing the principle of rule of law, and 4) Its influence on legal systems around the world."
    }
]

# Get completions for each input
outputs = [get_completion(build_input_prompt(item["text"])) for item in eval]

# Grade the completions
grades = [get_completion(build_grader_prompt(output, item["golden_answer"])) for output, item in zip(outputs, eval)]

# Print the summary quality score
print(f"Summary quality score: {grades[0]}")

### Пример 2: Проверка фактов

В этом примере мы воспользуемся Claude для проверки утверждения, а затем оценим точность его проверки. Это может быть полезно, когда вам нужно оценить способность модели различать точную и неточную информацию. Предоставляя рубрику, в которой описываются ключевые моменты, которые должны быть охвачены при правильной проверке фактов, мы можем автоматизировать процесс оценки и быстро оценить эффективность модели при выполнении задач по проверке фактов.

In [ ]:
# Function to build the input prompt for fact-checking
def build_input_prompt(claim):
    user_content = f"""Determine if the following claim is true or false:
    <claim>{claim}</claim>"""
    return [{"role": "user", "content": user_content}]

# Function to build the grader prompt for assessing fact-check accuracy
def build_grader_prompt(output, rubric):
    user_content = f"""Based on the following rubric, assess whether the fact-check is correct:
    <rubric>{rubric}</rubric>
    <fact-check>{output}</fact-check>"""
    return [{"role": "user", "content": user_content}]

# Define the evaluation data
eval = [
    {
        "claim": "The Great Wall of China is visible from space.",
        "golden_answer": "A correct fact-check should state that this claim is false. While the Great Wall is an impressive structure, it is not visible from space with the naked eye."
    }
]

# Get completions for each input
outputs = [get_completion(build_input_prompt(item["claim"])) for item in eval]

grades = []
for output, item in zip(outputs, eval):
    # Print the claim, fact-check, and rubric
    print(f"Claim: {item['claim']}\n")
    print(f"Fact-check: {output}]\n")
    print(f"Rubric: {item['golden_answer']}\n")
    
    # Grade the fact-check
    grader_prompt = build_grader_prompt(output, item["golden_answer"])
    grade = get_completion(grader_prompt)
    grades.append("correct" in grade.lower())

# Print the fact-checking accuracy
accuracy = sum(grades) / len(grades)
print(f"Fact-checking accuracy: {accuracy * 100}%")

### Пример 3: Анализ тона

В этом примере мы будем использовать Claude для анализа тона данного текста, а затем оценивать точность его анализа. Это может быть полезно, когда вам нужно оценить способность модели идентифицировать и интерпретировать эмоциональное содержание и отношения, выраженные в фрагменте текста. Предоставляя рубрику, в которой описываются ключевые аспекты тона, которые необходимо выявить, мы можем автоматизировать процесс оценки и быстро оценить эффективность модели при выполнении задач анализа тона.

In [ ]:
# Function to build the input prompt for tone analysis
def build_input_prompt(text):
    user_content = f"""Analyze the tone of the following text:
    <text>{text}</text>"""
    return [{"role": "user", "content": user_content}]

# Function to build the grader prompt for assessing tone analysis accuracy
def build_grader_prompt(output, rubric):
    user_content = f"""Assess the accuracy of the following tone analysis based on this rubric:
    <rubric>{rubric}</rubric>
    <tone-analysis>{output}</tone-analysis>"""
    return [{"role": "user", "content": user_content}]

# Define the evaluation data
eval = [
    {
        "text": "I can't believe they canceled the event at the last minute. This is completely unacceptable and unprofessional!",
        "golden_answer": "The tone analysis should identify the text as expressing frustration, anger, and disappointment. Key words like 'can't believe', 'last minute', 'unacceptable', and 'unprofessional' indicate strong negative emotions."
    }
]

# Get completions for each input
outputs = [get_completion(build_input_prompt(item["text"])) for item in eval]

# Grade the completions
grades = [get_completion(build_grader_prompt(output, item["golden_answer"])) for output, item in zip(outputs, eval)]

# Print the tone analysis quality
print(f"Tone analysis quality: {grades[0]}")

Эти примеры демонстрируют, как можно использовать оценку на основе кода, человека и модели для оценки моделей ИИ, таких как Клод, в различных задачах. Выбор метода оценки зависит от характера задачи и имеющихся ресурсов. Оценка на основе моделей предлагает многообещающий подход к автоматизации оценки сложных задач, которые в противном случае потребовали бы человеческого решения.